In [11]:
import numpy as np
import pandas as pd

# load datasets (this might take a few seconds)

train_data = pd.read_csv("mnist_train.csv", header=None).values
test_data = pd.read_csv("mnist_test.csv", header=None).values

# Shuffle the training data to ensure randomness

np.random.seed(42)
np.random.shuffle(train_data)

# Extract features (X) and labels (Y)
# We transpose X to have shape (784, m) which makes math easier later
# We also normalize pixels from 0-255 to 0.0-1.0
X_train = train_data[:, 1:] / 255.0
Y_train = train_data[:, 0]

X_test = test_data[:, 1:] / 255.0
Y_test = test_data[:, 0]
print(f"X_train shape: {X_train.shape} (784 pixels, {X_train.shape[1]} examples)")
print(f"Y_train shape: {Y_train.shape} (Labels)")
print(f"X_test shape:  {X_test.shape} (784 pixels, {X_test.shape[1]} examples)")
print(f"Y_test shape:  {Y_test.shape} (Labels)")

X_train shape: (60000, 784) (784 pixels, 784 examples)
Y_train shape: (60000,) (Labels)
X_test shape:  (10000, 784) (784 pixels, 784 examples)
Y_test shape:  (10000,) (Labels)


In [2]:
def init_params():
    W1 = np.random.rand(784, 128) - 0.5
    b1 = np.random.rand(1, 128) - 0.5
    W2 = np.random.rand(128, 10) - 0.5
    b2 = np.random.rand(1, 10) - 0.5
    return W1, b1, W2, b2

In [3]:
def ReLU(x):
    return np.maximum(x, 0)
    
def softmax(x):
    exp = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)

def forward_prop(x, W1, b1, W2, b2):
    step_1 = x @ W1 + b1 # enter to layer                  [x, 128]
    norm = ReLU(step_1) # out of the layer (activation)    [x, 128]
    step_2 = norm @ W2 + b2 # second enter to layer        [128, 10]
    logits = softmax(step_2) # out of the layer (probs)    [128, 10]
    return step_1, norm, step_2, logits  

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, int(Y.max()) + 1))
    one_hot_Y[np.arange(Y.size), Y.astype(int)] = 1
    return one_hot_Y

In [4]:
def backward_prop(X, Y, W2, step_1, norm, logits):
    m = X.shape[0]                                 # number of training examples (60,000 images)

    dlogits = logits - one_hot(Y)                  # error at the output layer 
    dW2 = (norm.T @ dlogits) / m                   # gradient flowing into W2
    db2 = np.sum(dlogits, axis=0) / m              # gradient flowing into b2

    dnorm = (dlogits @ W2.T) * (step_1 > 0)        # propagate error back through ReLU
    dW1 = (X.T @ dnorm) / m                        # gradient flowing into W1
    db1 = np.sum(dnorm, axis=0) / m                # gradient flowing into b1

    return dW1, db1, dW2, db2   

In [5]:
def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, lr):
    W1 = W1 - lr * dW1
    b1 = b1 - lr * db1
    W2 = W2 - lr * dW2
    b2 = b2 - lr * db2
    return W1, b1, W2, b2

In [6]:
W1, b1, W2, b2 = init_params() 

for i in range(1000):
    
    # forward
    step_1, norm, step_2, logits = forward_prop(X_train, W1, b1, W2, b2)

    # backward
    dW1, db1, dW2, db2 = backward_prop(X_train, Y_train, W2, step_1, norm, logits)

    # update
    W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, 0.1)

    if i % 100 == 0:
        predictions = np.argmax(logits, axis=1)
        accuracy = np.mean(predictions == Y_train)
        print(f"Step {i} | Accuracy: {accuracy:.2%}")

Step 0 | Accuracy: 8.46%
Step 100 | Accuracy: 83.02%
Step 200 | Accuracy: 86.89%
Step 300 | Accuracy: 88.66%
Step 400 | Accuracy: 89.74%
Step 500 | Accuracy: 90.48%
Step 600 | Accuracy: 91.10%
Step 700 | Accuracy: 91.60%
Step 800 | Accuracy: 92.00%
Step 900 | Accuracy: 92.34%


In [10]:
# test with unseen datas
step_1_test, norm_test, step_2_test, logits_test = forward_prop(X_test, W1, b1, W2, b2)
test_preds = np.argmax(logits_test, axis=1)
test_accuracy = np.mean(test_preds == Y_test)
print(f"Test Set Accuracy: {test_accuracy:.2%}")

Test Set Accuracy: 92.27%
